## Importación de librerías

In [1]:
!pip install gurobipy pandas

  Using cached gurobipy-13.0.3-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (16 kB)
  Using cached pandas-3.0.6-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.5.4-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached gurobipy-13.0.3-cp314-cp314-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (14.8 MB)
Using cached pandas-3.0.6-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (10.8 MB)
Using cached numpy-2.5.4-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]━━━━━━━━━━━ 2/3 [pandas]


In [2]:
import sys
import gurobipy as gp
from gurobipy import GRB
import time
import csv
import math
import random
import os
import pandas as pd

print('Python  :', sys.version.split()[0])
print('Gurobi  :', '.'.join(map(str, gp.gurobi.version())))

Python  : 3.14.4
Gurobi  : 13.0.3


### 1. Generación de Datos

In [3]:
# ==========================================
# 1. FUNCIÓN GENERADORA (CALIBRADA Y NO TRIVIAL)
# ==========================================
def generar_instancia_desde_cvrplib(nombre_instancia, datos_clientes, n_clientes=15, tipo_estres="capacidad"):
    N_C = list(range(1, n_clientes + 1))
    n_plus_1 = n_clientes + 1
    N = list(range(n_clientes + 2))
    D = [1, 2, 3]
    R = [1, 2, 3]

    coord_dep = (datos_clientes[0]['x'], datos_clientes[0]['y'])
    coordenadas = {0: coord_dep, n_plus_1: coord_dep}
    for i in N_C:
        coordenadas[i] = (datos_clientes[i]['x'], datos_clientes[i]['y'])

    A = [(i, j) for i in N for j in N if i != j and j != 0 and i != n_plus_1]

    td, tt = {}, {}
    for (i, j) in A:
        dist = math.sqrt((coordenadas[i][0] - coordenadas[j][0])**2 + (coordenadas[i][1] - coordenadas[j][1])**2)
        td[i, j] = round(dist, 2)
        tt[i, j] = round(dist, 2)

    q, st, a, b = {}, {}, {}, {}
    N_TW = []
    var_dia = {1: 1.0, 2: 1.15, 3: 0.90} # Fluctuación semanal

    for i in N_C:
        demanda_base = datos_clientes[i]['demanda']
        for t in D:
            q[i, t] = math.ceil(demanda_base * var_dia[t])
            st[i, t] = 10.0

        tw_inicio = datos_clientes[i]['inicio_tw']
        tw_fin = datos_clientes[i]['fin_tw']
        a[i] = tw_inicio
        b[i] = tw_inicio + max(30, (tw_fin - tw_inicio) * 0.35) if tipo_estres == "tiempo" else tw_fin
        N_TW.append(i)

    # 88% de ocupación para forzar Branch & Bound
    demanda_max_dia = max(sum(q[i, t] for i in N_C) for t in D)
    factor_apriete = 0.88 if tipo_estres == "capacidad" else 0.75
    Q = math.ceil((demanda_max_dia / len(R)) / factor_apriete)

    return {
        'nombre': nombre_instancia, 'A': A, 'N_C': N_C, 'N': N, 'N_TW': N_TW, 'R': R, 'D': D,
        'td': td, 'tt': tt, 'q': q, 'st': st, 'a': a, 'b': b,
        'a_W': datos_clientes[0]['inicio_tw'], 'b_W': datos_clientes[0]['fin_tw'],
        'Q': Q, 'M': 3000.0
    }

# ==========================================
# 2. BANCO DE INSTANCIAS (SOLOMON C101 Y R101)
# ==========================================
datos_solomon_c101 = [
    {'x': 40, 'y': 50, 'demanda': 0, 'inicio_tw': 0, 'fin_tw': 1236},
    {'x': 45, 'y': 68, 'demanda': 10, 'inicio_tw': 100, 'fin_tw': 250},
    {'x': 45, 'y': 70, 'demanda': 25, 'inicio_tw': 150, 'fin_tw': 300},
    {'x': 42, 'y': 66, 'demanda': 15, 'inicio_tw': 65,  'fin_tw': 180},
    {'x': 42, 'y': 68, 'demanda': 12, 'inicio_tw': 200, 'fin_tw': 350},
    {'x': 42, 'y': 65, 'demanda': 18, 'inicio_tw': 45,  'fin_tw': 150},
    {'x': 40, 'y': 69, 'demanda': 20, 'inicio_tw': 250, 'fin_tw': 400},
    {'x': 40, 'y': 66, 'demanda': 15, 'inicio_tw': 120, 'fin_tw': 240},
    {'x': 38, 'y': 68, 'demanda': 22, 'inicio_tw': 180, 'fin_tw': 320},
    {'x': 38, 'y': 70, 'demanda': 10, 'inicio_tw': 300, 'fin_tw': 450},
    {'x': 35, 'y': 66, 'demanda': 14, 'inicio_tw': 80,  'fin_tw': 200},
    {'x': 35, 'y': 69, 'demanda': 16, 'inicio_tw': 220, 'fin_tw': 360},
    {'x': 25, 'y': 85, 'demanda': 20, 'inicio_tw': 350, 'fin_tw': 500},
    {'x': 22, 'y': 75, 'demanda': 28, 'inicio_tw': 50,  'fin_tw': 160},
    {'x': 22, 'y': 85, 'demanda': 12, 'inicio_tw': 280, 'fin_tw': 420},
    {'x': 20, 'y': 80, 'demanda': 35, 'inicio_tw': 150, 'fin_tw': 280}
]

datos_solomon_r101 = [
    {'x': 35, 'y': 35, 'demanda': 0, 'inicio_tw': 0, 'fin_tw': 1000},
    {'x': 41, 'y': 49, 'demanda': 15, 'inicio_tw': 80,  'fin_tw': 180},
    {'x': 35, 'y': 17, 'demanda': 18, 'inicio_tw': 40,  'fin_tw': 120},
    {'x': 55, 'y': 45, 'demanda': 22, 'inicio_tw': 120, 'fin_tw': 220},
    {'x': 55, 'y': 20, 'demanda': 10, 'inicio_tw': 60,  'fin_tw': 150},
    {'x': 15, 'y': 30, 'demanda': 12, 'inicio_tw': 30,  'fin_tw': 110},
    {'x': 25, 'y': 30, 'demanda': 16, 'inicio_tw': 90,  'fin_tw': 200},
    {'x': 20, 'y': 50, 'demanda': 20, 'inicio_tw': 150, 'fin_tw': 260},
    {'x': 10, 'y': 40, 'demanda': 14, 'inicio_tw': 110, 'fin_tw': 210},
    {'x': 55, 'y': 60, 'demanda': 12, 'inicio_tw': 200, 'fin_tw': 310},
    {'x': 30, 'y': 60, 'demanda': 15, 'inicio_tw': 140, 'fin_tw': 240},
    {'x': 20, 'y': 65, 'demanda': 18, 'inicio_tw': 180, 'fin_tw': 290},
    {'x': 50, 'y': 35, 'demanda': 25, 'inicio_tw': 70,  'fin_tw': 160},
    {'x': 30, 'y': 25, 'demanda': 10, 'inicio_tw': 40,  'fin_tw': 130},
    {'x': 15, 'y': 10, 'demanda': 20, 'inicio_tw': 60,  'fin_tw': 150},
    {'x': 60, 'y': 15, 'demanda': 14, 'inicio_tw': 100, 'fin_tw': 200}
]

banco_instancias = {
    "Instancia1_Clustered": {"datos": datos_solomon_c101, "n_clientes": 15},
    "Instancia2_Dispersa": {"datos": datos_solomon_r101, "n_clientes": 15}
}
print("Datos y banco de instancias listos para el bucle.")

Datos y banco de instancias listos para el bucle.


### 2. Definición del Framework Matemático

In [4]:
class ConsistentVRPFramework:
    def __init__(self, data):
        self.data = data
        self.model = None
        self.X, self.Z, self.W, self.dW, self.rW = {}, {}, {}, {}, {}

    def build_model(self):
        self.model = gp.Model("CVRP_Consistent")
        N_C, N, N_TW, R, D = self.data['N_C'], self.data['N'], self.data['N_TW'], self.data['R'], self.data['D']
        td, tt, q, st = self.data['td'], self.data['tt'], self.data['q'], self.data['st']
        a, b, a_W, b_W, Q, M = self.data['a'], self.data['b'], self.data['a_W'], self.data['b_W'], self.data['Q'], self.data['M']
        A = self.data['A']

        self.X = self.model.addVars(A, D, R, vtype=GRB.BINARY, name="X")
        self.Z = self.model.addVars(N_C, R, vtype=GRB.BINARY, name="Z")
        self.W = self.model.addVars(N_C, D, vtype=GRB.CONTINUOUS, lb=0.0, name="W")
        self.dW = self.model.addVars(R, vtype=GRB.CONTINUOUS, lb=0.0, name="dW")
        self.rW = self.model.addVars(R, D, vtype=GRB.CONTINUOUS, lb=0.0, name="rW")

        self.model.setObjective(gp.quicksum(td[i, j] * self.X[i, j, t, r] for (i, j) in A for t in D for r in R), GRB.MINIMIZE)

        for i in N_C:
            self.model.addConstr(gp.quicksum(self.Z[i, r] for r in R) == 1, name=f"AsignacionUnica_{i}")

        for j in N_C:
            for r in R:
                for t in D:
                    self.model.addConstr(gp.quicksum(self.X[i, j, t, r] for i in N if (i, j) in A) == self.Z[j, r], name=f"FlujoEntrada_{j}_{r}_{t}")
        for i in N_C:
            for r in R:
                for t in D:
                    self.model.addConstr(gp.quicksum(self.X[i, j, t, r] for j in N if (i, j) in A) == self.Z[i, r], name=f"FlujoSalida_{i}_{r}_{t}")
        for r in R:
            for t in D:
                 self.model.addConstr(gp.quicksum(self.X[0, j, t, r] for j in N if (0, j) in A) == 1, name=f"SalidaDep_{r}_{t}")

        for r in R:
            for t in D:
                for i in N_C:
                    for j in N_C:
                        if i != j and (i, j) in A:
                            self.model.addConstr(self.W[i, t] + st[i, t] + tt[i, j] <= self.W[j, t] + M * (1 - self.X[i, j, t, r]), name=f"BigM_{i}_{j}_{r}_{t}")

        for i in N_TW:
            for t in D:
                self.model.addConstr(self.W[i, t] >= a[i], name=f"TW_Inicio_{i}_{t}")
                self.model.addConstr(self.W[i, t] <= b[i], name=f"TW_Fin_{i}_{t}")

        n_plus_1 = max(N)
        for j in N_C:
            for r in R:
                for t in D:
                    self.model.addConstr(self.dW[r] + tt[0, j] <= self.W[j, t] + M * (1 - self.X[0, j, t, r]), name=f"CoordSalida_{j}_{r}_{t}")
        for i in N_C:
            for r in R:
                for t in D:
                    self.model.addConstr(self.W[i, t] + st[i, t] + tt[i, n_plus_1] <= self.rW[r, t] + M * (1 - self.X[i, n_plus_1, t, r]), name=f"CoordRet_{i}_{r}_{t}")

        for r in R:
            self.model.addConstr(self.dW[r] >= a_W, name=f"Ap_Dep_{r}")
            for t in D:
                self.model.addConstr(self.rW[r, t] <= b_W, name=f"Ci_Dep_{r}_{t}")
                self.model.addConstr(gp.quicksum(q[i, t] * self.Z[i, r] for i in N_C) <= Q, name=f"Capacidad_{r}_{t}")

        self.model.update()

    def set_base_params(self):
        self.model.resetParams() # LIMPIEZA TOTAL PARA EVITAR EFECTO CASCADA
        self.model.setParam('Threads', 1)
        self.model.setParam('TimeLimit', 600)
        self.model.setParam('OutputFlag', 0)

    def run_relaxation(self):
        self.set_base_params()
        relaxed = self.model.relax()
        relaxed.optimize()
        return relaxed.ObjVal if relaxed.Status == GRB.OPTIMAL else None

    def run_configuration(self, config_name, params_dict):
        self.model.reset()
        self.set_base_params()
        for p, v in params_dict.items():
            self.model.setParam(p, v)

        start = time.time()
        self.model.optimize()
        t = time.time() - start

        if self.model.Status in [GRB.OPTIMAL, GRB.TIME_LIMIT] and self.model.SolCount > 0:
            return self.model.ObjVal, self.model.MIPGap * 100, t
        return None, None, t

    def run_tuning(self, tune_time_limit=300):
        print(" -> Ejecutando Gurobi Tuning...")
        self.model.reset()
        self.set_base_params()
        self.model.setParam('TuneTimeLimit', tune_time_limit)

        start = time.time()
        self.model.tune()

        if self.model.TuneResultCount > 0:
            self.model.getTuneResult(0)
            self.model.optimize()
            t = time.time() - start
            if self.model.SolCount > 0:
                return self.model.ObjVal, self.model.MIPGap * 100, t
        return None, None, time.time() - start

    def run_sensitivity_analysis(self, filename="sensibilidad.csv"):
      if self.model.SolCount == 0:
          print("    [!] No hay solución factible para análisis de sensibilidad.")
          return

      fixed_model = self.model.fixed()
      fixed_model.setParam('OutputFlag', 0)
      fixed_model.setParam('Method', 1)   # Dual Simplex
      fixed_model.setParam('Presolve', 0) # REQUERIDO para habilitar SARHSLow/SARHSUp
      fixed_model.optimize()

      if fixed_model.Status == GRB.OPTIMAL:
          with open(filename, mode='w', newline='', encoding='utf-8') as file:
              writer = csv.writer(file)
              writer.writerow(["Restriccion", "Precio_Sombra_Pi", "SARHSLow", "SARHSUp"])
              for r in self.data['R']:
                  for t in self.data['D']:
                      constr = fixed_model.getConstrByName(f"Capacidad_{r}_{t}")
                      if constr:
                          writer.writerow([f"Cap_{r}_{t}", f"{constr.Pi:.4f}", f"{constr.SARHSLow:.2f}", f"{constr.SARHSUp:.2f}"])
          print(f"    [OK] Sensibilidad guardada en {filename}")

    def exportar_resultados_csv(self, z_LP, res_milp, filename="resultados.csv"):
        with open(filename, mode='w', newline='', encoding='utf-8') as file:
            writer = csv.writer(file)
            writer.writerow(["Configuracion", "Valor_Objetivo", "Gap_%", "Tiempo_s", "Brecha_Integralidad_%"])

            # Validación explícita de z_LP para evitar errores de formato
            lp_str = f"{z_LP:.4f}" if z_LP is not None else "N/A"
            writer.writerow(["LP Relax", lp_str, "-", "-", "-"])

            z_MILP_base = res_milp.get("Baseline", (None, None, None))[0]

            for config, (obj, gap, run_time) in res_milp.items():
                # Validación estricta usando 'is not None' y previniendo división por cero
                if config == "Baseline" and z_LP is not None and z_MILP_base is not None and z_MILP_base != 0:
                    brecha = f"{((z_MILP_base - z_LP) / z_MILP_base * 100):.2f}"
                else:
                    brecha = "-"

                # Validación explícita para el resto de métricas
                obj_str = f"{obj:.4f}" if obj is not None else "N/A"
                gap_str = f"{gap:.2f}" if gap is not None else "-"

                writer.writerow([config, obj_str, gap_str, f"{run_time:.2f}", brecha])

### 3. Construcción y Ejecución de Experimentos

In [5]:
# ==========================================
# 3. BUCLE DE EXPERIMENTACIÓN AUTOMÁTICA
# ==========================================
tipos_de_estres = ["capacidad"]

for nombre_inst, info in banco_instancias.items():
    for estres in tipos_de_estres:
        sufijo_archivo = f"{nombre_inst}_{estres}"
        print(f"\n==============================================")
        print(f"PROCESANDO: {sufijo_archivo.upper()}")
        print(f"==============================================")

        try:
            # 1. Construir Instancia y Modelo
            data_instancia = generar_instancia_desde_cvrplib(
                nombre_instancia=sufijo_archivo,
                datos_clientes=info["datos"],
                n_clientes=info["n_clientes"],
                tipo_estres=estres
            )
            framework = ConsistentVRPFramework(data_instancia)
            framework.build_model()

            # 2. Relajación LP
            print(" -> Calculando LP Relax...")
            z_LP = framework.run_relaxation()
            print(f"    Valor LP: {z_LP:.2f}" if z_LP else "    Sin solución LP")

            # 3. Configuraciones MILP
            configs = {
                "Baseline": {},
                "No_Cuts": {'Cuts': 0},
                "No_Presolve": {'Presolve': 0},
                "MIPFocus_1": {'MIPFocus': 1},
                "MIPFocus_3": {'MIPFocus': 3},
                "No_Heuristics": {'Heuristics': 0}
            }

            resultados_milp = {}
            for nombre_config, params in configs.items():
                print(f" -> Corriendo MILP: {nombre_config}...")
                obj, gap, t = framework.run_configuration(nombre_config, params)
                resultados_milp[nombre_config] = (obj, gap, t)

            # 4. Gurobi Tuning
            obj_tune, gap_tune, t_tune = framework.run_tuning(tune_time_limit=300)
            resultados_milp["Tuning_Gurobi"] = (obj_tune, gap_tune, t_tune)

            # 5. Análisis de Sensibilidad (Se fuerza correr el Baseline limpio primero)
            print(" -> Preparando Análisis de Sensibilidad...")
            framework.run_configuration("Baseline", {'TimeLimit': 600})
            framework.run_sensitivity_analysis(f"sensibilidad_{sufijo_archivo}.csv")

            # 6. Guardar Resultados
            framework.exportar_resultados_csv(z_LP, resultados_milp, f"resultados_{sufijo_archivo}.csv")
            print(f"[¡OK!] Experimento {sufijo_archivo} completado y guardado con éxito.")

        except Exception as e:
            # Evita que un error mate el código durante la noche
            print(f"\n[ERROR CRÍTICO] Falló el procesamiento de {sufijo_archivo}.")
            print(f"Detalle técnico: {e}")
            print("Saltando automáticamente a la siguiente instancia...")

print("\n¡FIN DE TODOS LOS EXPERIMENTOS!")


PROCESANDO: INSTANCIA1_CLUSTERED_CAPACIDAD
Set parameter Username
Set parameter LicenseID to value 2874581
Academic license - for non-commercial use only - expires 2027-10-14
 -> Calculando LP Relax...
Reset all parameters
Set parameter Threads to value 1
Set parameter TimeLimit to value 600
    Valor LP: 121.50
 -> Corriendo MILP: Baseline...
Set parameter Threads to value 1
Set parameter TimeLimit to value 600
 -> Corriendo MILP: No_Cuts...
Set parameter Threads to value 1
Set parameter TimeLimit to value 600
 -> Corriendo MILP: No_Presolve...
Set parameter Threads to value 1
Set parameter TimeLimit to value 600
 -> Corriendo MILP: MIPFocus_1...
Set parameter Threads to value 1
Set parameter TimeLimit to value 600
 -> Corriendo MILP: MIPFocus_3...
Set parameter Threads to value 1
Set parameter TimeLimit to value 600
 -> Corriendo MILP: No_Heuristics...
Set parameter Threads to value 1
Set parameter TimeLimit to value 600
 -> Ejecutando Gurobi Tuning...
Set parameter Threads to value

### 4. Visualización de resultados

In [8]:
# ==========================================
# 4. VISUALIZACIÓN DE RESULTADOS
# ==========================================
# Cambia a "Instancia2_Dispersa_capacidad" para ver la otra
nombre_ver = "Instancia1_Clustered_capacidad"

print(f"--- Resultados: {nombre_ver} ---")
archivo_resultados = f"resultados_{nombre_ver}.csv"

if os.path.exists(archivo_resultados):
    display(pd.read_csv(archivo_resultados))
else:
    print(f"[!] No se encontró el archivo: {archivo_resultados}")

print(f"\n--- Sensibilidad: {nombre_ver} ---")
archivo_sensibilidad = f"sensibilidad_{nombre_ver}.csv"

if os.path.exists(archivo_sensibilidad):
    display(pd.read_csv(archivo_sensibilidad).head(10))
else:
    print(f"[!] No se generó el archivo de sensibilidad para {nombre_ver}.")

--- Resultados: Instancia1_Clustered_capacidad ---


,Configuracion,Valor_Objetivo,Gap_%,Tiempo_s,Brecha_Integralidad_%
0,LP Relax,121.50,-,-,-
1,Baseline,519.03,13.22,600.00,76.59
2,No_Cuts,519.03,23.40,600.00,-
3,No_Presolve,519.03,25.71,600.00,-
4,MIPFocus_1,519.03,20.72,600.00,-
5,MIPFocus_3,519.03,10.52,600.00,-
6,No_Heuristics,520.38,11.32,600.00,-
7,Tuning_Gurobi,NaN,-,600.19,-



--- Sensibilidad: Instancia1_Clustered_capacidad ---


,Restriccion,Precio_Sombra_Pi,SARHSLow,SARHSUp
0,Cap_1_1,0.0,95.0,inf
1,Cap_1_2,0.0,112.0,inf
2,Cap_1_3,0.0,88.0,inf
3,Cap_2_1,0.0,95.0,inf
4,Cap_2_2,0.0,111.0,inf
5,Cap_2_3,0.0,87.0,inf
6,Cap_3_1,0.0,82.0,inf
7,Cap_3_2,0.0,97.0,inf
8,Cap_3_3,0.0,75.0,inf


### 5.Analisis de los datos

#### Resultados de los experimentos

In [2]:
import pandas as pd

In [5]:
BASE_EXPERIMENTOS = r"resultados_experimentos/"

In [6]:
resultados_1 = pd.read_csv(BASE_EXPERIMENTOS + "resultados_Instancia1_Clustered_capacidad.csv")
resultados_2 = pd.read_csv(BASE_EXPERIMENTOS + "resultados_Instancia2_Dispersa_capacidad.csv")

In [7]:
resultados_1

,Configuracion,Valor_Objetivo,Gap_%,Tiempo_s,Brecha_Integralidad_%
0,LP Relax,121.50,-,-,-
1,Baseline,519.03,13.22,600.00,76.59
2,No_Cuts,519.03,23.40,600.00,-
3,No_Presolve,519.03,25.71,600.00,-
4,MIPFocus_1,519.03,20.72,600.00,-
5,MIPFocus_3,519.03,10.52,600.00,-
6,No_Heuristics,520.38,11.32,600.00,-
7,Tuning_Gurobi,NaN,-,600.19,-


In [8]:
resultados_2

,Configuracion,Valor_Objetivo,Gap_%,Tiempo_s,Brecha_Integralidad_%
0,LP Relax,571.29,-,-,-
1,Baseline,899.64,0.01,8.07,36.50
2,No_Cuts,899.64,0.00,27.96,-
3,No_Presolve,899.64,0.00,127.60,-
4,MIPFocus_1,899.64,0.00,11.38,-
5,MIPFocus_3,899.64,0.01,14.35,-
6,No_Heuristics,899.64,0.00,7.21,-
7,Tuning_Gurobi,899.64,0.01,293.66,-


#### Análisis de sensibilidad

In [10]:
sensibilidad_1 = pd.read_csv(BASE_EXPERIMENTOS + "sensibilidad_Instancia1_Clustered_capacidad.csv")
sensibilidad_2 = pd.read_csv(BASE_EXPERIMENTOS + "sensibilidad_Instancia2_Dispersa_capacidad.csv")

In [11]:
sensibilidad_1

,Restriccion,Precio_Sombra_Pi,SARHSLow,SARHSUp
0,Cap_1_1,0.0,95.0,inf
1,Cap_1_2,0.0,112.0,inf
2,Cap_1_3,0.0,88.0,inf
3,Cap_2_1,0.0,95.0,inf
4,Cap_2_2,0.0,111.0,inf
5,Cap_2_3,0.0,87.0,inf
6,Cap_3_1,0.0,82.0,inf
7,Cap_3_2,0.0,97.0,inf
8,Cap_3_3,0.0,75.0,inf


In [12]:
sensibilidad_2

,Restriccion,Precio_Sombra_Pi,SARHSLow,SARHSUp
0,Cap_1_1,0.0,86.0,inf
1,Cap_1_2,0.0,102.0,inf
2,Cap_1_3,0.0,79.0,inf
3,Cap_2_1,0.0,91.0,inf
4,Cap_2_2,0.0,107.0,inf
5,Cap_2_3,0.0,84.0,inf
6,Cap_3_1,0.0,64.0,inf
7,Cap_3_2,0.0,75.0,inf
8,Cap_3_3,0.0,59.0,inf
